In [0]:
from pyspark.sql import functions as F, types as T

catalog = dbutils.widgets.get("catalog")

results = []

def check(name, severity, ok, actual, expected, detail_pass, detail_fail):
    results.append((name, severity, str(actual), str(expected), "PASS" if ok else "FAIL", detail_pass if ok else detail_fail))

# --- 1) all eight source tables ingested (non-empty bronze tables) ---
source_tables = ["region", "nation", "customer", "supplier", "part", "partsupp", "orders", "lineitem"]
for t in source_tables:
    n = spark.table(f"{catalog}.bronze.{t}").count()
    check(
        f"Source ingested: bronze.{t}", "CRITICAL", n > 0, n, "> 0",
        "Bronze table is non-empty",
        "Bronze table is EMPTY — ingestion did not run or wrote nothing",
    )

# --- 2) primary keys populated and unique (silver tables + gold dimensions, one scan each) ---
pk_map = {
    "region": ["r_regionkey"],
    "nation": ["n_nationkey"],
    "customer": ["c_custkey"],
    "supplier": ["s_suppkey"],
    "part": ["p_partkey"],
    "partsupp": ["ps_partkey", "ps_suppkey"],
    "order": ["o_orderkey"],
    "lineitem": ["l_orderkey", "l_linenumber"],
}
for tbl, pk_cols in pk_map.items():
    null_cond = " OR ".join(f"{c} IS NULL" for c in pk_cols)
    r = spark.sql(f"""
        SELECT
            COUNT(*) AS total,
            COUNT(DISTINCT {", ".join(pk_cols)}) AS distinct_keys,
            SUM(CASE WHEN {null_cond} THEN 1 ELSE 0 END) AS null_keys
        FROM {catalog}.silver.{tbl}
    """).collect()[0]
    check(
        f"PK populated: silver.{tbl} ({' + '.join(pk_cols)})", "CRITICAL", (r["null_keys"] or 0) == 0,
        r["null_keys"] or 0, 0,
        "Primary key fully populated",
        f"{r['null_keys']} row(s) with NULL primary key",
    )
    check(
        f"PK unique: silver.{tbl} ({' + '.join(pk_cols)})", "CRITICAL", r["distinct_keys"] == r["total"],
        r["distinct_keys"], r["total"],
        "Primary key unique",
        f"{r['total'] - r['distinct_keys']} duplicate primary key value(s)",
    )

gold_dims = [
    ("dim_customer", "customer_key"),
    ("dim_product", "product_key"),
    ("dim_supplier", "supplier_key"),
    ("dim_geography", "geography_key"),
]
for dim, key in gold_dims:
    r = spark.sql(f"""
        SELECT
            COUNT(*) AS total,
            COUNT(DISTINCT {key}) AS distinct_keys,
            SUM(CASE WHEN {key} IS NULL THEN 1 ELSE 0 END) AS null_keys
        FROM {catalog}.gold.{dim}
    """).collect()[0]
    check(
        f"PK populated: gold.{dim}.{key}", "CRITICAL", (r["null_keys"] or 0) == 0,
        r["null_keys"] or 0, 0,
        "Dimension key fully populated",
        f"{r['null_keys']} row(s) with NULL dimension key",
    )
    check(
        f"PK unique: gold.{dim}.{key}", "CRITICAL", r["distinct_keys"] == r["total"],
        r["distinct_keys"], r["total"],
        "Dimension key unique",
        f"{r['total'] - r['distinct_keys']} duplicate dimension key(s)",
    )

# --- 3) fact: no null keys + documented measure rules ) ---
m = spark.sql(f"""
    SELECT
        COUNT(*) AS fact_rows,
        SUM(CASE WHEN order_key IS NULL OR line_number IS NULL THEN 1 ELSE 0 END) AS bad_fact_pk,
        SUM(CASE WHEN customer_key IS NULL THEN 1 ELSE 0 END) AS null_customer_key,
        SUM(CASE WHEN product_key IS NULL THEN 1 ELSE 0 END) AS null_product_key,
        SUM(CASE WHEN supplier_key IS NULL THEN 1 ELSE 0 END) AS null_supplier_key,
        SUM(CASE WHEN customer_nation_key IS NULL THEN 1 ELSE 0 END) AS null_customer_nation_key,
        SUM(CASE WHEN supplier_nation_key IS NULL THEN 1 ELSE 0 END) AS null_supplier_nation_key,
        SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) AS bad_quantity,
        SUM(CASE WHEN extended_price <= 0 THEN 1 ELSE 0 END) AS bad_extended_price,
        SUM(CASE WHEN discount_rate < 0 OR discount_rate > 1 THEN 1 ELSE 0 END) AS bad_discount,
        SUM(CASE WHEN tax_rate < 0 OR tax_rate > 1 THEN 1 ELSE 0 END) AS bad_tax,
        SUM(CASE WHEN order_date IS NULL OR order_date NOT BETWEEN '1992-01-01' AND '1998-12-31' THEN 1 ELSE 0 END) AS bad_order_date,
        SUM(CASE WHEN receipt_date < ship_date THEN 1 ELSE 0 END) AS bad_ship_timeline
    FROM {catalog}.gold.fact_sales
""").collect()[0]

fact_checks = [
    ("bad_fact_pk", "Fact grain populated: (order_key, line_number) not null in gold.fact_sales", "Every fact row has both key columns"),
    ("null_customer_key", "No null keys: gold.fact_sales.customer_key", "Every fact row references a customer"),
    ("null_product_key", "No null keys: gold.fact_sales.product_key", "Every fact row references a product"),
    ("null_supplier_key", "No null keys: gold.fact_sales.supplier_key", "Every fact row references a supplier"),
    ("null_customer_nation_key", "No null keys: gold.fact_sales.customer_nation_key", "Every fact row has customer geography"),
    ("null_supplier_nation_key", "No null keys: gold.fact_sales.supplier_nation_key", "Every fact row has supplier geography"),
    ("bad_quantity", "Measure rule: quantity > 0 in gold.fact_sales", "Documented rule: quantity is positive"),
    ("bad_extended_price", "Measure rule: extended_price > 0 in gold.fact_sales", "Documented rule: extended price is positive"),
    ("bad_discount", "Measure rule: 0 <= discount_rate <= 1 in gold.fact_sales", "Documented rule: discount rate within [0, 1]"),
    ("bad_tax", "Measure rule: 0 <= tax_rate <= 1 in gold.fact_sales", "Documented rule: tax rate within [0, 1]"),
    ("bad_order_date", "Measure rule: order_date within 1992-01-01..1998-12-31", "Documented rule: order dates within the TPCH window"),
    ("bad_ship_timeline", "Measure rule: receipt_date >= ship_date in gold.fact_sales", "Documented rule: a line cannot be received before it shipped"),
]
for field, name, rule in fact_checks:
    bad = m[field] or 0
    check(name, "CRITICAL", bad == 0, bad, 0, rule, f"{bad} row(s) violate: {rule}")

# --- 4) Gold reconciles with Silver ---
rec = spark.sql(f"""
    SELECT
        (SELECT COUNT(*) FROM {catalog}.gold.fact_sales) AS gold_rows,
        (SELECT COUNT(*) FROM {catalog}.silver.lineitem) AS silver_rows,
        (SELECT SUM(extended_price) FROM {catalog}.gold.fact_sales) AS gold_revenue,
        (SELECT SUM(l_extendedprice) FROM {catalog}.silver.lineitem) AS silver_revenue,
        (SELECT SUM(annual_revenue) FROM {catalog}.gold.vw_annual_sales_kpis) AS gold_net_revenue,
        (SELECT SUM(l_extendedprice * (1 - l_discount)) FROM {catalog}.silver.lineitem) AS silver_net_revenue
""").collect()[0]

check(
    "Reconciliation: fact row count vs silver.lineitem", "CRITICAL",
    rec["gold_rows"] == rec["silver_rows"], rec["gold_rows"], rec["silver_rows"],
    "Gold row count matches Silver",
    "Gold row count differs from Silver — records lost or duplicated in the Gold build",
)
check(
    "Reconciliation: gross revenue vs silver.lineitem", "CRITICAL",
    rec["gold_revenue"] == rec["silver_revenue"], rec["gold_revenue"], rec["silver_revenue"],
    "Gold gross revenue matches Silver",
    "Gold gross revenue differs from the independent Silver calculation",
)
check(
    "Reconciliation: net revenue (vw_annual_sales_kpis) vs silver.lineitem", "CRITICAL",
    rec["gold_net_revenue"] == rec["silver_net_revenue"], rec["gold_net_revenue"], rec["silver_net_revenue"],
    "KPI view net revenue matches Silver",
    "KPI view net revenue differs from the independent Silver calculation",
)

# --- 5) rejected-record counts reported ---
rejected_total = 0
for bronze_name, silver_name in [
    ("region", "region"), ("nation", "nation"), ("customer", "customer"), ("supplier", "supplier"),
    ("part", "part"), ("partsupp", "partsupp"), ("orders", "order"), ("lineitem", "lineitem"),
]:
    n = spark.table(f"{catalog}.silver.{silver_name}_quarantine").count()
    rejected_total += n
    results.append(
        (f"Rejected records: {bronze_name}", "REPORT", str(n), "reported only", "INFO",
         f"Rows preserved in silver.{silver_name}_quarantine")
    )
results.append(
    ("Rejected records: TOTAL", "REPORT", str(rejected_total), "reported only", "INFO",
     "Total quarantined across all eight tables")
)

# --- persist results, then enforce the gate ---
schema = T.StructType(
    [
        T.StructField("check_name", T.StringType()),
        T.StructField("severity", T.StringType()),
        T.StructField("actual", T.StringType()),
        T.StructField("expected", T.StringType()),
        T.StructField("status", T.StringType()),
        T.StructField("detail", T.StringType()),
    ]
)
dq = spark.createDataFrame(results, schema).withColumn("generated_at", F.current_timestamp())
dq.write.mode("overwrite").saveAsTable(f"{catalog}.gold.data_quality_results")

critical_fails = [r for r in results if r[1] == "CRITICAL" and r[4] == "FAIL"]
print(f"DATA QUALITY GATE: {len(results)} checks -> {sum(1 for r in results if r[4] == 'PASS')} PASS, {len(critical_fails)} CRITICAL FAIL, {rejected_total} rejected records reported")
for fl in critical_fails:
    print("CRITICAL FAIL:", fl[0], "|", fl[5])
print("Results table:", f"{catalog}.gold.data_quality_results")

display(dq.orderBy("severity", "check_name"))

if critical_fails:
    raise Exception(
        f"DATA QUALITY GATE FAILED — {len(critical_fails)} critical check(s) failed. "
        f"Details in {catalog}.gold.data_quality_results. First failures: {[r[0] for r in critical_fails[:5]]}"
    )

print("DATA QUALITY GATE PASSED — safe to publish Gold.")